# 03 - Privacy experiments

This notebook tests subject-identification leakage. The prediction target is `subject_ids`, not the motor imagery label.

## Contents

1. Setup and data loading
2. Feature extraction
3. Attacker models
4. Experiment runner
5. Subject subsets
6. Privacy experiments
7. Save results

## 1. Setup

Mount Drive, import packages, and set the input/output paths.

In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Drive mount skipped. This is fine outside Colab.")

Mounted at /content/drive


In [2]:
from pathlib import Path
import gc
import random

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)


set_seed()

In [3]:
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_PATH = DRIVE_ROOT / "URV_Datasets" / "physionet_mi_lr_imagery_subjects_1_50_with_runs.npz"
RESULTS_DIR = DRIVE_ROOT / "URV_Datasets" / "results"
RESULTS_CSV = RESULTS_DIR / "physionet_privacy_results_clean.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Data:", DATA_PATH)
print("Results:", RESULTS_CSV)

Data: /content/drive/MyDrive/URV_Datasets/physionet_mi_lr_imagery_subjects_1_50_with_runs.npz
Results: /content/drive/MyDrive/URV_Datasets/results/physionet_privacy_results_clean.csv


## 2. Load the dataset

The privacy experiments use the run-aware 1-50 subject file.

In [4]:
def load_dataset(path):
    data = np.load(path, allow_pickle=False)
    required = ["X", "y", "subject_ids", "run_ids"]
    missing = [key for key in required if key not in data.files]
    if missing:
        raise KeyError(f"Missing keys: {missing}")

    X = data["X"].astype("float32")
    y_task = data["y"].astype("int64")
    subject_ids = data["subject_ids"].astype("int64")
    run_ids = data["run_ids"].astype("int64")

    if not (len(X) == len(y_task) == len(subject_ids) == len(run_ids)):
        raise ValueError("X, y, subject_ids, and run_ids do not have the same length.")

    return X, y_task, subject_ids, run_ids


X, y_task, subject_ids, run_ids = load_dataset(DATA_PATH)

print("X:", X.shape)
print("Task labels:", dict(zip(*np.unique(y_task, return_counts=True))))
print("Runs:", dict(zip(*np.unique(run_ids, return_counts=True))))
print("Subjects:", len(np.unique(subject_ids)))

expected_subjects = set(range(1, 51))
actual_subjects = set(np.unique(subject_ids).tolist())
print("Missing subjects:", sorted(expected_subjects - actual_subjects))
print("Extra subjects:", sorted(actual_subjects - expected_subjects))

X: (2245, 64, 641)
Task labels: {np.int64(0): np.int64(1132), np.int64(1): np.int64(1113)}
Runs: {np.int64(4): np.int64(750), np.int64(8): np.int64(747), np.int64(12): np.int64(748)}
Subjects: 50
Missing subjects: []
Extra subjects: []


## 3. Preprocessing helpers

Keep subject subsets and optionally normalize each trial/channel.

In [5]:
def filter_subjects(X, y_task, subject_ids, run_ids, start, end):
    mask = (subject_ids >= start) & (subject_ids <= end)
    return X[mask], y_task[mask], subject_ids[mask], run_ids[mask]


def zscore_per_trial_channel(X, eps=1e-6):
    mean = X.mean(axis=2, keepdims=True)
    std = X.std(axis=2, keepdims=True)
    return ((X - mean) / (std + eps)).astype("float32")

## 4. Feature extraction

The attackers do not use labels as features. Features are computed from the EEG signal: summary statistics per channel and optional channel correlations.

In [6]:
def extract_privacy_features(X, include_connectivity=True):
    X = X.astype("float32")

    features = [
        X.mean(axis=2),
        X.std(axis=2),
        np.log(np.var(X, axis=2) + 1e-6),
        X.min(axis=2),
        X.max(axis=2),
        np.median(X, axis=2),
    ]

    # Channel-correlation features capture spatial relationships between EEG channels.
    if include_connectivity:
        n_trials, n_channels, _ = X.shape
        upper_idx = np.triu_indices(n_channels, k=1)
        corr_features = np.empty((n_trials, len(upper_idx[0])), dtype="float32")

        for i in range(n_trials):
            corr = np.corrcoef(X[i])
            corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
            corr_features[i] = corr[upper_idx]

        features.append(corr_features)

    return np.concatenate(features, axis=1).astype("float32")

## 5. Attacker models

Use one linear attacker and one nonlinear attacker.

In [7]:
def build_attackers():
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=5000, solver="lbfgs", n_jobs=-1, random_state=SEED)),
        ]),
        "Random Forest": RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            random_state=SEED,
            n_jobs=-1,
        ),
    }

## 6. Experiment runner

Random splits measure the easier enrolled-user setting. Run-held-out splits train on runs 4 and 8 and test on run 12 for the same enrolled subjects.

In [8]:
results = []


def run_privacy_experiment(
    X,
    subject_ids,
    run_ids,
    experiment,
    subjects_label,
    split_type,
    normalize,
    include_connectivity,
    train_runs=(4, 8),
    test_run=12,
    test_size=0.20,
    print_details=False,
):
    set_seed()

    # Optional per-trial normalization removes simple amplitude scale effects.
    X_proc = zscore_per_trial_channel(X) if normalize else X.astype("float32")
    X_features = extract_privacy_features(X_proc, include_connectivity=include_connectivity)

    if split_type == "random_trial":
        X_train, X_test, y_train, y_test = train_test_split(
            X_features,
            subject_ids,
            test_size=test_size,
            random_state=SEED,
            stratify=subject_ids,
        )
        split_label = f"random stratified trial split, test_size={test_size}"
        train_runs_label = "mixed"
        test_runs_label = "mixed"

    elif split_type == "run_held_out":
        # Same enrolled subjects, different run.
        train_mask = np.isin(run_ids, list(train_runs))
        test_mask = run_ids == test_run

        X_train = X_features[train_mask]
        X_test = X_features[test_mask]
        y_train = subject_ids[train_mask]
        y_test = subject_ids[test_mask]

        train_subjects = set(np.unique(y_train).tolist())
        test_subjects = set(np.unique(y_test).tolist())
        if train_subjects != test_subjects:
            raise ValueError("Run-held-out privacy needs the same enrolled subjects in train and test.")

        split_label = f"run-held-out: train runs {list(train_runs)}, test run {test_run}"
        train_runs_label = ",".join(map(str, train_runs))
        test_runs_label = str(test_run)

    else:
        raise ValueError("split_type must be 'random_trial' or 'run_held_out'")

    chance_accuracy = 1 / len(np.unique(subject_ids))
    normalization = "per-trial per-channel z-score" if normalize else "none"
    feature_set = "summary + channel correlation" if include_connectivity else "summary only"

    print("\n", experiment)
    print("Features:", X_features.shape, "|", feature_set)
    print("Train:", X_train.shape, "Test:", X_test.shape)
    print("Train subjects:", len(np.unique(y_train)), "Test subjects:", len(np.unique(y_test)))
    print("Chance:", chance_accuracy)

    for model_name, attacker in build_attackers().items():
        attacker.fit(X_train, y_train)
        y_pred = attacker.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        f1_weighted = f1_score(y_test, y_pred, average="weighted")
        leakage = acc / chance_accuracy

        print(model_name, "accuracy:", acc, "balanced:", bal_acc, "leakage:", leakage)

        if print_details:
            print(classification_report(y_test, y_pred))
            print(confusion_matrix(y_test, y_pred))

        results.append({
            "experiment": experiment,
            "dataset": "PhysioNet Motor Imagery",
            "subjects": subjects_label,
            "target": "subject identification",
            "label": "subject_ids",
            "split": split_label,
            "model": model_name,
            "accuracy": acc,
            "balanced_accuracy": bal_acc,
            "f1_weighted": f1_weighted,
            "chance_accuracy": chance_accuracy,
            "leakage_over_chance": leakage,
            "n_train": len(y_train),
            "n_test": len(y_test),
            "train_runs": train_runs_label,
            "test_runs": test_runs_label,
            "normalization": normalization,
            "features": feature_set,
        })

    gc.collect()

## 7. Subject subsets

Build the 1-10 and 1-50 subject subsets used below.

In [9]:
X_10, y_task_10, subject_ids_10, run_ids_10 = filter_subjects(X, y_task, subject_ids, run_ids, 1, 10)
X_50, y_task_50, subject_ids_50, run_ids_50 = filter_subjects(X, y_task, subject_ids, run_ids, 1, 50)

print("Subjects 1-10:", X_10.shape, "chance =", 1 / len(np.unique(subject_ids_10)))
print("Subjects 1-50:", X_50.shape, "chance =", 1 / len(np.unique(subject_ids_50)))

Subjects 1-10: (450, 64, 641) chance = 0.1
Subjects 1-50: (2245, 64, 641) chance = 0.02


## 8. Subjects 1-10

Small-subject privacy checks.

In [10]:
run_privacy_experiment(
    X_10,
    subject_ids_10,
    run_ids_10,
    experiment="Privacy Exp 1: subjects 1-10 random trial",
    subjects_label="1-10",
    split_type="random_trial",
    normalize=False,
    include_connectivity=True,
)

run_privacy_experiment(
    X_10,
    subject_ids_10,
    run_ids_10,
    experiment="Privacy Exp 2: subjects 1-10 run-held-out",
    subjects_label="1-10",
    split_type="run_held_out",
    normalize=False,
    include_connectivity=True,
)


 Privacy Exp 1: subjects 1-10 random trial
Features: (450, 2400) | summary + channel correlation
Train: (360, 2400) Test: (90, 2400)
Train subjects: 10 Test subjects: 10
Chance: 0.1
Logistic Regression accuracy: 1.0 balanced: 1.0 leakage: 10.0
Random Forest accuracy: 1.0 balanced: 1.0 leakage: 10.0

 Privacy Exp 2: subjects 1-10 run-held-out
Features: (450, 2400) | summary + channel correlation
Train: (300, 2400) Test: (150, 2400)
Train subjects: 10 Test subjects: 10
Chance: 0.1
Logistic Regression accuracy: 1.0 balanced: 1.0 leakage: 10.0
Random Forest accuracy: 0.9933333333333333 balanced: 0.9933333333333334 leakage: 9.933333333333332


## 9. Subjects 1-50

Main privacy leakage experiments.

In [11]:
run_privacy_experiment(
    X_50,
    subject_ids_50,
    run_ids_50,
    experiment="Privacy Exp 3: subjects 1-50 random trial",
    subjects_label="1-50",
    split_type="random_trial",
    normalize=False,
    include_connectivity=True,
)

run_privacy_experiment(
    X_50,
    subject_ids_50,
    run_ids_50,
    experiment="Privacy Exp 4: subjects 1-50 run-held-out",
    subjects_label="1-50",
    split_type="run_held_out",
    normalize=False,
    include_connectivity=True,
)

run_privacy_experiment(
    X_50,
    subject_ids_50,
    run_ids_50,
    experiment="Privacy Exp 5: subjects 1-50 run-held-out normalized",
    subjects_label="1-50",
    split_type="run_held_out",
    normalize=True,
    include_connectivity=True,
)


 Privacy Exp 3: subjects 1-50 random trial
Features: (2245, 2400) | summary + channel correlation
Train: (1796, 2400) Test: (449, 2400)
Train subjects: 50 Test subjects: 50
Chance: 0.02
Logistic Regression accuracy: 0.9977728285077951 balanced: 0.9977777777777778 leakage: 49.88864142538976
Random Forest accuracy: 0.9977728285077951 balanced: 0.9977777777777778 leakage: 49.88864142538976

 Privacy Exp 4: subjects 1-50 run-held-out
Features: (2245, 2400) | summary + channel correlation
Train: (1497, 2400) Test: (748, 2400)
Train subjects: 50 Test subjects: 50
Chance: 0.02
Logistic Regression accuracy: 0.9852941176470589 balanced: 0.9852380952380954 leakage: 49.26470588235294
Random Forest accuracy: 0.9919786096256684 balanced: 0.992 leakage: 49.59893048128342

 Privacy Exp 5: subjects 1-50 run-held-out normalized
Features: (2245, 2400) | summary + channel correlation
Train: (1497, 2400) Test: (748, 2400)
Train subjects: 50 Test subjects: 50
Chance: 0.02
Logistic Regression accuracy: 0.9

## 10. Conservative check

Run-held-out, normalized, and no connectivity features.

In [12]:
RUN_CONSERVATIVE_CHECK = True

if RUN_CONSERVATIVE_CHECK:
    run_privacy_experiment(
        X_50,
        subject_ids_50,
        run_ids_50,
        experiment="Privacy Exp 6: subjects 1-50 run-held-out normalized, no connectivity",
        subjects_label="1-50",
        split_type="run_held_out",
        normalize=True,
        include_connectivity=False,
    )


 Privacy Exp 6: subjects 1-50 run-held-out normalized, no connectivity
Features: (2245, 384) | summary only
Train: (1497, 384) Test: (748, 384)
Train subjects: 50 Test subjects: 50
Chance: 0.02
Logistic Regression accuracy: 0.6871657754010695 balanced: 0.687904761904762 leakage: 34.35828877005348
Random Forest accuracy: 0.8435828877005348 balanced: 0.8433333333333333 leakage: 42.17914438502674


## 11. Save results

Write the privacy results to CSV.

In [13]:
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
display(results_df)
print("Saved:", RESULTS_CSV)

,experiment,dataset,subjects,target,label,split,model,accuracy,balanced_accuracy,f1_weighted,chance_accuracy,leakage_over_chance,n_train,n_test,train_runs,test_runs,normalization,features
0,Privacy Exp 1: subjects 1-10 random trial,PhysioNet Motor Imagery,1-10,subject identification,subject_ids,"random stratified trial split, test_size=0.2",Logistic Regression,1.000000,1.000000,1.000000,0.10,10.000000,360,90,mixed,mixed,none,summary + channel correlation
1,Privacy Exp 1: subjects 1-10 random trial,PhysioNet Motor Imagery,1-10,subject identification,subject_ids,"random stratified trial split, test_size=0.2",Random Forest,1.000000,1.000000,1.000000,0.10,10.000000,360,90,mixed,mixed,none,summary + channel correlation
2,Privacy Exp 2: subjects 1-10 run-held-out,PhysioNet Motor Imagery,1-10,subject identification,subject_ids,"run-held-out: train runs [4, 8], test run 12",Logistic Regression,1.000000,1.000000,1.000000,0.10,10.000000,300,150,"4,8",12,none,summary + channel correlation
3,Privacy Exp 2: subjects 1-10 run-held-out,PhysioNet Motor Imagery,1-10,subject identification,subject_ids,"run-held-out: train runs [4, 8], test run 12",Random Forest,0.993333,0.993333,0.993326,0.10,9.933333,300,150,"4,8",12,none,summary + channel correlation
4,Privacy Exp 3: subjects 1-50 random trial,PhysioNet Motor Imagery,1-50,subject identification,subject_ids,"random stratified trial split, test_size=0.2",Logistic Regression,0.997773,0.997778,0.997766,0.02,49.888641,1796,449,mixed,mixed,none,summary + channel correlation
5,Privacy Exp 3: subjects 1-50 random trial,PhysioNet Motor Imagery,1-50,subject identification,subject_ids,"random stratified trial split, test_size=0.2",Random Forest,0.997773,0.997778,0.997766,0.02,49.888641,1796,449,mixed,mixed,none,summary + channel correlation
6,Privacy Exp 4: subjects 1-50 run-held-out,PhysioNet Motor Imagery,1-50,subject identification,subject_ids,"run-held-out: train runs [4, 8], test run 12",Logistic Regression,0.985294,0.985238,0.985067,0.02,49.264706,1497,748,"4,8",12,none,summary + channel correlation
7,Privacy Exp 4: subjects 1-50 run-held-out,PhysioNet Motor Imagery,1-50,subject identification,subject_ids,"run-held-out: train runs [4, 8], test run 12",Random Forest,0.991979,0.992000,0.991879,0.02,49.598930,1497,748,"4,8",12,none,summary + channel correlation
8,Privacy Exp 5: subjects 1-50 run-held-out norm...,PhysioNet Motor Imagery,1-50,subject identification,subject_ids,"run-held-out: train runs [4, 8], test run 12",Logistic Regression,0.986631,0.986667,0.986446,0.02,49.331551,1497,748,"4,8",12,per-trial per-channel z-score,summary + channel correlation
9,Privacy Exp 5: subjects 1-50 run-held-out norm...,PhysioNet Motor Imagery,1-50,subject identification,subject_ids,"run-held-out: train runs [4, 8], test run 12",Random Forest,0.991979,0.992000,0.991879,0.02,49.598930,1497,748,"4,8",12,per-trial per-channel z-score,summary + channel correlation


Saved: /content/drive/MyDrive/URV_Datasets/results/physionet_privacy_results_clean.csv
